In [1]:
import sys

import xgboost

print("Python:", sys.executable)
print("XGBoost:", xgboost.__version__)
print("All imports successful!")

Python: c:\Users\hyd\Desktop\FloodSentinels_part1\FloodSentinel_Latest\venv\Scripts\python.exe
XGBoost: 3.2.0
All imports successful!


In [4]:
import pandas as pd

DATA_PATH = "../data/raw/water_sensor_dataset (2).csv"

water_df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", water_df.shape)
print("Columns:")
print(water_df.columns.tolist())

Dataset loaded successfully!
Shape: (350400, 19)
Columns:
['sensor_id', 'location_name', 'latitude', 'longitude', 'pipe_diameter_mm', 'pipe_material', 'pipe_slope', 'elevation_m', 'timestamp', 'month', 'water_level_cm', 'rise_rate_cm_per_15min', 'rainfall_mm_per_hr', 'cumulative_rain_6hr_mm', 'cumulative_rain_24hr_mm', 'battery_voltage', 'signal_strength_dbm', 'sensor_status', 'overflow_flag']


In [5]:
# Check the target column and missing values

print("Missing values:")
print(water_df.isnull().sum())

print("\nOverflow flag distribution:")
print(water_df["overflow_flag"].value_counts(dropna=False))

Missing values:
sensor_id                     0
location_name                 0
latitude                      0
longitude                     0
pipe_diameter_mm              0
pipe_material                 0
pipe_slope                    0
elevation_m                   0
timestamp                     0
month                         0
water_level_cm             5270
rise_rate_cm_per_15min     5270
rainfall_mm_per_hr         5270
cumulative_rain_6hr_mm     5270
cumulative_rain_24hr_mm    5270
battery_voltage               0
signal_strength_dbm        5270
sensor_status                 0
overflow_flag              5270
dtype: int64

Overflow flag distribution:
overflow_flag
0.0    343389
NaN      5270
1.0      1741
Name: count, dtype: int64


In [6]:
# Make sure data is ordered correctly for each sensor
water_df["timestamp"] = pd.to_datetime(water_df["timestamp"])

water_df = water_df.sort_values(
    ["sensor_id", "timestamp"]
).reset_index(drop=True)

# Overflow status at the next 15-minute reading
water_df["overflow_next_15min"] = (
    water_df.groupby("sensor_id")["overflow_flag"].shift(-1)
)

# Create early-warning target
water_df["overflow_onset_next_15min"] = pd.NA

valid_transition = (
    water_df["overflow_flag"].notna() &
    water_df["overflow_next_15min"].notna()
)

water_df.loc[valid_transition, "overflow_onset_next_15min"] = (
    (
        (water_df.loc[valid_transition, "overflow_flag"] == 0) &
        (water_df.loc[valid_transition, "overflow_next_15min"] == 1)
    ).astype(int)
)

print("Target distribution:")
print(water_df["overflow_onset_next_15min"].value_counts(dropna=False))

Target distribution:
overflow_onset_next_15min
0       338416
<NA>     10470
1         1514
Name: count, dtype: int64


In [7]:

import numpy as np
import pandas as pd

water_df["timestamp"] = pd.to_datetime(water_df["timestamp"])
water_df = water_df.sort_values(
    ["sensor_id", "timestamp"]
).reset_index(drop=True)
water_df["overflow_next_15min"] = (
    water_df.groupby("sensor_id")["overflow_flag"].shift(-1)
)
water_df["overflow_next_30min"] = (
    water_df.groupby("sensor_id")["overflow_flag"].shift(-2)
)
water_df["overflow_onset_next_15min"] = np.nan

valid_15 = (
    water_df["overflow_flag"].notna()
    & water_df["overflow_next_15min"].notna()
)

water_df.loc[valid_15, "overflow_onset_next_15min"] = (
    (
        (water_df.loc[valid_15, "overflow_flag"] == 0)
        &
        (water_df.loc[valid_15, "overflow_next_15min"] == 1)
    ).astype(int)
)

water_df["overflow_onset_next_30min"] = np.nan
valid_30 = (
    water_df["overflow_flag"].notna()
    & water_df["overflow_next_15min"].notna()
    & water_df["overflow_next_30min"].notna()
)
water_df.loc[valid_30, "overflow_onset_next_30min"] = (
    (
        (water_df.loc[valid_30, "overflow_flag"] == 0)
        &
        (
            (water_df.loc[valid_30, "overflow_next_15min"] == 1)
            |
            (water_df.loc[valid_30, "overflow_next_30min"] == 1)
        )
    ).astype(int)
)
print("========== 15-MINUTE TARGET ==========")
print(water_df["overflow_onset_next_15min"].value_counts(dropna=False))

print("\n========== 30-MINUTE TARGET ==========")
print(water_df["overflow_onset_next_30min"].value_counts(dropna=False))

========== 15-MINUTE TARGET ==========
overflow_onset_next_15min
0.0    338416
NaN     10470
1.0      1514
Name: count, dtype: int64

========== 30-MINUTE TARGET ==========
overflow_onset_next_30min
0.0    332211
NaN     15602
1.0      2587
Name: count, dtype: int64


In [8]:

base_df = water_df[
    water_df["overflow_flag"] == 0
].copy()

print("Rows with current NO overflow:", len(base_df))




model_15 = base_df[
    base_df["overflow_onset_next_15min"].notna()
].copy()

print("\n15-minute dataset shape:", model_15.shape)

print("\n15-minute target distribution:")
print(
    model_15["overflow_onset_next_15min"]
    .value_counts()
)



model_30 = base_df[
    base_df["overflow_onset_next_30min"].notna()
].copy()

print("\n30-minute dataset shape:", model_30.shape)

print("\n30-minute target distribution:")
print(
    model_30["overflow_onset_next_30min"]
    .value_counts()
)

Rows with current NO overflow: 343389

15-minute dataset shape: (338212, 23)

15-minute target distribution:
overflow_onset_next_15min
0.0    336698
1.0      1514
Name: count, dtype: int64

30-minute dataset shape: (333107, 23)

30-minute target distribution:
overflow_onset_next_30min
0.0    330520
1.0      2587
Name: count, dtype: int64


In [9]:
# ============================================
# STEP 3: Select features
# ============================================

feature_columns = [
    "water_level_cm",
    "rise_rate_cm_per_15min",
    "rainfall_mm_per_hr",
    "cumulative_rain_6hr_mm",
    "cumulative_rain_24hr_mm",
    "pipe_diameter_mm",
    "pipe_material",
    "pipe_slope",
    "elevation_m",
    "month"
]

target_15 = "overflow_onset_next_15min"
target_30 = "overflow_onset_next_30min"


# Create X and y for 15-minute model
X_15 = model_15[feature_columns].copy()
y_15 = model_15[target_15].astype(int)

# Create X and y for 30-minute model
X_30 = model_30[feature_columns].copy()
y_30 = model_30[target_30].astype(int)


print("15-minute X shape:", X_15.shape)
print("15-minute y shape:", y_15.shape)

print("\n30-minute X shape:", X_30.shape)
print("30-minute y shape:", y_30.shape)

print("\nFeatures:")
print(feature_columns)

15-minute X shape: (338212, 10)
15-minute y shape: (338212,)

30-minute X shape: (333107, 10)
30-minute y shape: (333107,)

Features:
['water_level_cm', 'rise_rate_cm_per_15min', 'rainfall_mm_per_hr', 'cumulative_rain_6hr_mm', 'cumulative_rain_24hr_mm', 'pipe_diameter_mm', 'pipe_material', 'pipe_slope', 'elevation_m', 'month']


In [10]:
# ============================================
# STEP 2: Prepare datasets for XGBoost
# ============================================

# Keep only rows where there is currently NO overflow.
# This makes the problem a true "early warning" problem.
base_df = water_df[
    water_df["overflow_flag"] == 0
].copy()

print("Rows with current NO overflow:", len(base_df))


# ------------------------------------------------
# 15-minute modeling dataset
# ------------------------------------------------

model_15 = base_df[
    base_df["overflow_onset_next_15min"].notna()
].copy()

print("\n15-minute dataset shape:", model_15.shape)

print("\n15-minute target distribution:")
print(
    model_15["overflow_onset_next_15min"]
    .value_counts()
)


# ------------------------------------------------
# 30-minute modeling dataset
# ------------------------------------------------

model_30 = base_df[
    base_df["overflow_onset_next_30min"].notna()
].copy()

print("\n30-minute dataset shape:", model_30.shape)

print("\n30-minute target distribution:")
print(
    model_30["overflow_onset_next_30min"]
    .value_counts()
)

Rows with current NO overflow: 343389

15-minute dataset shape: (338212, 23)

15-minute target distribution:
overflow_onset_next_15min
0.0    336698
1.0      1514
Name: count, dtype: int64

30-minute dataset shape: (333107, 23)

30-minute target distribution:
overflow_onset_next_30min
0.0    330520
1.0      2587
Name: count, dtype: int64


In [11]:

feature_columns = [
    "water_level_cm",
    "rise_rate_cm_per_15min",
    "rainfall_mm_per_hr",
    "cumulative_rain_6hr_mm",
    "cumulative_rain_24hr_mm",
    "pipe_diameter_mm",
    "pipe_material",
    "pipe_slope",
    "elevation_m",
    "month"
]

target_15 = "overflow_onset_next_15min"
target_30 = "overflow_onset_next_30min"


X_15 = model_15[feature_columns].copy()
y_15 = model_15[target_15].astype(int)


X_30 = model_30[feature_columns].copy()
y_30 = model_30[target_30].astype(int)


print("15-minute X shape:", X_15.shape)
print("15-minute y shape:", y_15.shape)

print("\n30-minute X shape:", X_30.shape)
print("30-minute y shape:", y_30.shape)

print("\nSelected features:")
for feature in feature_columns:
    print("-", feature)

15-minute X shape: (338212, 10)
15-minute y shape: (338212,)

30-minute X shape: (333107, 10)
30-minute y shape: (333107,)

Selected features:
- water_level_cm
- rise_rate_cm_per_15min
- rainfall_mm_per_hr
- cumulative_rain_6hr_mm
- cumulative_rain_24hr_mm
- pipe_diameter_mm
- pipe_material
- pipe_slope
- elevation_m
- month


In [12]:

# Make sure timestamp is datetime
model_15["timestamp"] = pd.to_datetime(model_15["timestamp"])
model_30["timestamp"] = pd.to_datetime(model_30["timestamp"])

# Sort chronologically
model_15 = model_15.sort_values("timestamp").reset_index(drop=True)
model_30 = model_30.sort_values("timestamp").reset_index(drop=True)

print("15-minute time range:")
print(model_15["timestamp"].min(), "to", model_15["timestamp"].max())

print("\n30-minute time range:")
print(model_30["timestamp"].min(), "to", model_30["timestamp"].max())

15-minute time range:
2026-01-01 00:00:00 to 2026-12-31 23:30:00

30-minute time range:
2026-01-01 00:00:00 to 2026-12-31 23:15:00


In [14]:


train_15 = model_15[
    model_15["timestamp"].dt.month <= 7
].copy()

validation_15 = model_15[
    model_15["timestamp"].dt.month == 8
].copy()

test_15 = model_15[
    model_15["timestamp"].dt.month >= 9
].copy()



train_30 = model_30[
    model_30["timestamp"].dt.month <= 7
].copy()

validation_30 = model_30[
    model_30["timestamp"].dt.month == 8
].copy()

test_30 = model_30[
    model_30["timestamp"].dt.month >= 9
].copy()



print("========== 15-MINUTE SPLIT ==========")

print("Train      :", train_15.shape)
print("Validation :", validation_15.shape)
print("Test       :", test_15.shape)

print("\n========== 30-MINUTE SPLIT ==========")

print("Train      :", train_30.shape)
print("Validation :", validation_30.shape)
print("Test       :", test_30.shape)

========== 15-MINUTE SPLIT ==========
Train      : (196753, 23)
Validation : (28224, 23)
Test       : (113235, 23)

========== 30-MINUTE SPLIT ==========
Train      : (193790, 23)
Validation : (27812, 23)
Test       : (111505, 23)


In [15]:
# ============================================
# STEP 4C: Check positive/negative counts
# ============================================

print("========== 15-MINUTE TARGETS ==========")

print("\nTrain:")
print(train_15["overflow_onset_next_15min"].value_counts())

print("\nValidation:")
print(validation_15["overflow_onset_next_15min"].value_counts())

print("\nTest:")
print(test_15["overflow_onset_next_15min"].value_counts())


print("\n========== 30-MINUTE TARGETS ==========")

print("\nTrain:")
print(train_30["overflow_onset_next_30min"].value_counts())

print("\nValidation:")
print(validation_30["overflow_onset_next_30min"].value_counts())

print("\nTest:")
print(test_30["overflow_onset_next_30min"].value_counts())

========== 15-MINUTE TARGETS ==========

Train:
overflow_onset_next_15min
0.0    196133
1.0       620
Name: count, dtype: int64

Validation:
overflow_onset_next_15min
0.0    27644
1.0      580
Name: count, dtype: int64

Test:
overflow_onset_next_15min
0.0    112921
1.0       314
Name: count, dtype: int64

========== 30-MINUTE TARGETS ==========

Train:
overflow_onset_next_30min
0.0    192725
1.0      1065
Name: count, dtype: int64

Validation:
overflow_onset_next_30min
0.0    26829
1.0      983
Name: count, dtype: int64

Test:
overflow_onset_next_30min
0.0    110966
1.0       539
Name: count, dtype: int64


In [16]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Fit ONLY on training data
pipe_train = encoder.fit_transform(
    train_15[["pipe_material"]]
)

pipe_validation = encoder.transform(
    validation_15[["pipe_material"]]
)

pipe_test = encoder.transform(
    test_15[["pipe_material"]]
)

print("Categories learned:")
print(encoder.categories_)

Categories learned:
[array(['Cast Iron', 'Concrete', 'PVC'], dtype=object)]


In [ ]:

from sklearn.preprocessing import OneHotEncoder

encoder_15 = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Learn categories ONLY from 15-min training data
pipe_train_15 = encoder_15.fit_transform(
    train_15[["pipe_material"]]
)

# Transform validation and test using the same encoder
pipe_validation_15 = encoder_15.transform(
    validation_15[["pipe_material"]]
)

pipe_test_15 = encoder_15.transform(
    test_15[["pipe_material"]]
)



encoder_30 = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Learn categories ONLY from 30-min training data
pipe_train_30 = encoder_30.fit_transform(
    train_30[["pipe_material"]]
)

# Transform validation and test
pipe_validation_30 = encoder_30.transform(
    validation_30[["pipe_material"]]
)

pipe_test_30 = encoder_30.transform(
    test_30[["pipe_material"]]
)


print("15-minute categories:")
print(encoder_15.categories_)

print("\n30-minute categories:")
print(encoder_30.categories_)

15-minute categories:
[array(['Cast Iron', 'Concrete', 'PVC'], dtype=object)]

30-minute categories:
[array(['Cast Iron', 'Concrete', 'PVC'], dtype=object)]


In [19]:

numeric_features = [
    "water_level_cm",
    "rise_rate_cm_per_15min",
    "rainfall_mm_per_hr",
    "cumulative_rain_6hr_mm",
    "cumulative_rain_24hr_mm",
    "pipe_diameter_mm",
    "pipe_slope",
    "elevation_m",
    "month"
]

X_train_15_numeric = train_15[numeric_features].to_numpy()
X_validation_15_numeric = validation_15[numeric_features].to_numpy()
X_test_15_numeric = test_15[numeric_features].to_numpy()

X_train_15_final = np.hstack(
    [X_train_15_numeric, pipe_train_15]
)

X_validation_15_final = np.hstack(
    [X_validation_15_numeric, pipe_validation_15]
)

X_test_15_final = np.hstack(
    [X_test_15_numeric, pipe_test_15]
)


X_train_30_numeric = train_30[numeric_features].to_numpy()
X_validation_30_numeric = validation_30[numeric_features].to_numpy()
X_test_30_numeric = test_30[numeric_features].to_numpy()

X_train_30_final = np.hstack(
    [X_train_30_numeric, pipe_train_30]
)

X_validation_30_final = np.hstack(
    [X_validation_30_numeric, pipe_validation_30]
)

X_test_30_final = np.hstack(
    [X_test_30_numeric, pipe_test_30]
)
print("15-minute final shapes:")
print("Train      :", X_train_15_final.shape)
print("Validation :", X_validation_15_final.shape)
print("Test       :", X_test_15_final.shape)

print("\n30-minute final shapes:")
print("Train      :", X_train_30_final.shape)
print("Validation :", X_validation_30_final.shape)
print("Test       :", X_test_30_final.shape)

15-minute final shapes:
Train      : (196753, 12)
Validation : (28224, 12)
Test       : (113235, 12)

30-minute final shapes:
Train      : (193790, 12)
Validation : (27812, 12)
Test       : (111505, 12)


In [ ]:

# 15-minute targets
y_train_15 = train_15["overflow_onset_next_15min"].astype(int).to_numpy()
y_validation_15 = validation_15["overflow_onset_next_15min"].astype(int).to_numpy()
y_test_15 = test_15["overflow_onset_next_15min"].astype(int).to_numpy()


# 30-minute targets
y_train_30 = train_30["overflow_onset_next_30min"].astype(int).to_numpy()
y_validation_30 = validation_30["overflow_onset_next_30min"].astype(int).to_numpy()
y_test_30 = test_30["overflow_onset_next_30min"].astype(int).to_numpy()


# Check shapes
print("15-minute target shapes:")
print("Train      :", y_train_15.shape)
print("Validation :", y_validation_15.shape)
print("Test       :", y_test_15.shape)

print("\n30-minute target shapes:")
print("Train      :", y_train_30.shape)
print("Validation :", y_validation_30.shape)
print("Test       :", y_test_30.shape)

15-minute target shapes:
Train      : (196753,)
Validation : (28224,)
Test       : (113235,)

30-minute target shapes:
Train      : (193790,)
Validation : (27812,)
Test       : (111505,)


In [ ]:


negative_15 = (y_train_15 == 0).sum()
positive_15 = (y_train_15 == 1).sum()

scale_pos_weight_15 = negative_15 / positive_15

negative_30 = (y_train_30 == 0).sum()
positive_30 = (y_train_30 == 1).sum()

scale_pos_weight_30 = negative_30 / positive_30


print("========== 15-MINUTE MODEL ==========")
print("NO samples :", negative_15)
print("YES samples:", positive_15)
print("scale_pos_weight:", scale_pos_weight_15)


print("\n========== 30-MINUTE MODEL ==========")
print("NO samples :", negative_30)
print("YES samples:", positive_30)
print("scale_pos_weight:", scale_pos_weight_30)

========== 15-MINUTE MODEL ==========
NO samples : 196133
YES samples: 620
scale_pos_weight: 316.34354838709675

========== 30-MINUTE MODEL ==========
NO samples : 192725
YES samples: 1065
scale_pos_weight: 180.96244131455398


In [22]:

from xgboost import XGBClassifier

xgb_15 = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight_15,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

print("Training 15-minute XGBoost model...")

xgb_15.fit(
    X_train_15_final,
    y_train_15,
    eval_set=[
        (X_train_15_final, y_train_15),
        (X_validation_15_final, y_validation_15)
    ],
    verbose=False
)

print("15-minute XGBoost training completed!")

Training 15-minute XGBoost model...
15-minute XGBoost training completed!


In [23]:
# Generate probability predictions
validation_prob_15 = xgb_15.predict_proba(X_validation_15_final)[:, 1]
test_prob_15 = xgb_15.predict_proba(X_test_15_final)[:, 1]

print("Validation probability shape:", validation_prob_15.shape)
print("Test probability shape:", test_prob_15.shape)

print("\nFirst 10 validation probabilities:")
print(validation_prob_15[:10])

print("\nFirst 10 test probabilities:")
print(test_prob_15[:10])

Validation probability shape: (28224,)
Test probability shape: (113235,)

First 10 validation probabilities:
[3.2156706e-06 2.1606625e-06 1.9626773e-06 4.9834839e-06 1.5600597e-05
 1.7893359e-04 4.9809132e-06 7.5140820e-06 1.2641741e-05 2.5937363e-05]

First 10 test probabilities:
[9.95246683e-06 1.37685274e-05 5.72190902e-06 1.37649113e-04
 2.11375118e-06 2.45228807e-06 6.78702054e-06 2.95312088e-06
 1.03391185e-05 4.05661303e-05]


In [24]:
import numpy as np
from sklearn.metrics import precision_recall_curve

# Calculate precision, recall and thresholds on validation data
precision_15, recall_15, thresholds_15 = precision_recall_curve(
    y_validation_15,
    validation_prob_15
)

# Calculate F1-score for every threshold
f1_scores_15 = (
    2 * precision_15[:-1] * recall_15[:-1]
    / (precision_15[:-1] + recall_15[:-1] + 1e-10)
)

# Find threshold with maximum F1
best_index_15 = np.argmax(f1_scores_15)
best_threshold_15 = thresholds_15[best_index_15]

print("Best threshold:", best_threshold_15)
print("Best validation F1:", f1_scores_15[best_index_15])
print("Precision at best threshold:", precision_15[best_index_15])
print("Recall at best threshold:", recall_15[best_index_15])

Best threshold: 0.98720145
Best validation F1: 0.5389312976605792
Precision at best threshold: 0.48356164383561645
Recall at best threshold: 0.6086206896551725


In [25]:
best_threshold_15 = 0.98720145

In [26]:
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# Convert test probabilities into YES/NO predictions
test_pred_15 = (
    test_prob_15 >= best_threshold_15
).astype(int)

# Calculate evaluation metrics
test_precision_15 = precision_score(y_test_15, test_pred_15)
test_recall_15 = recall_score(y_test_15, test_pred_15)
test_f1_15 = f1_score(y_test_15, test_pred_15)

# PR-AUC
test_pr_auc_15 = average_precision_score(
    y_test_15,
    test_prob_15
)

# Confusion matrix
cm_15 = confusion_matrix(
    y_test_15,
    test_pred_15
)

print("15-Minute XGBoost Test Results")
print("--------------------------------")

print("Precision:", test_precision_15)
print("Recall:", test_recall_15)
print("F1-score:", test_f1_15)
print("PR-AUC:", test_pr_auc_15)

print("\nConfusion Matrix:")
print(cm_15)

15-Minute XGBoost Test Results
--------------------------------
Precision: 0.41081081081081083
Recall: 0.7261146496815286
F1-score: 0.5247410817031071
PR-AUC: 0.5133677268170126

Confusion Matrix:
[[112594    327]
 [    86    228]]


In [27]:
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

# Thresholds we want to compare
thresholds_to_test = [0.99, 0.95, 0.90, 0.80, 0.70, 0.60, 0.50]

results = []

for threshold in thresholds_to_test:

    # Convert probability to YES/NO
    predictions = (test_prob_15 >= threshold).astype(int)

    # Calculate metrics
    precision = precision_score(y_test_15, predictions, zero_division=0)
    recall = recall_score(y_test_15, predictions, zero_division=0)
    f1 = f1_score(y_test_15, predictions, zero_division=0)

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        y_test_15,
        predictions
    ).ravel()

    results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })

# Create a table
threshold_results = pd.DataFrame(results)

print(threshold_results)

   Threshold  Precision    Recall        F1   TP    FP   FN      TN
0       0.99   0.427105  0.662420  0.519351  208   279  106  112642
1       0.95   0.332108  0.863057  0.479646  271   545   43  112376
2       0.90   0.290323  0.888535  0.437647  279   682   35  112239
3       0.80   0.254192  0.917197  0.398065  288   845   26  112076
4       0.70   0.241210  0.939490  0.383865  295   928   19  111993
5       0.60   0.226961  0.949045  0.366318  298  1015   16  111906
6       0.50   0.214183  0.952229  0.349708  299  1097   15  111824
